# Lesson 1: Your first model

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/Marmara-Comp-Pathology/comppath-course/blob/main/lessons/01-first-model/lesson.ipynb) [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Marmara-Comp-Pathology/comppath-course/blob/main/lessons/01-first-model/lesson.ipynb)

In this lesson you will train a computer to tell **lymph node metastasis** from **normal lymph node tissue** on small H&E patches.
You will not write any code. You will run the cells one by one, watch what happens, and look at the results the way a pathologist would.

Nothing is explained in depth yet, and that is on purpose. We play the *whole game* first, then spend the rest of the course
opening each box to see how it works.

## What you will learn

- Run a Jupyter notebook on a free cloud GPU
- Train a deep learning model that classifies H&E patches as *tumor* or *normal*
- Read a confusion matrix in terms you already know: sensitivity and specificity
- Look at the model's worst mistakes and ask *why*, as you would at a consensus meeting
- Name one reason the results look better than they really are (we fix this in lesson 5)

**Time:** about 4 to 6 hours over two weeks: half for the lesson, half for the homework.
Running the notebook itself takes about 15 minutes.

## Before you start

**On Kaggle (recommended)**

1. Create a free account at [kaggle.com](https://www.kaggle.com) and verify your phone number
   (Settings → Phone verification). Without it you get neither a GPU nor internet access.
2. Open the competition page [Histopathologic Cancer Detection](https://www.kaggle.com/competitions/histopathologic-cancer-detection)
   and accept its rules (the "Late Submission" or "Join Competition" button). This gives you access to the images.
3. Click the **Open in Kaggle** badge at the top of this page. Kaggle makes your own copy of the notebook.
4. In the right-hand panel:
   - **Settings → Accelerator → GPU T4 x2** (or P100)
   - **Settings → Internet → On**
   - **Add Input → Competitions →** search *histopathologic-cancer-detection* → **Add**

**On Google Colab (fallback)**

1. Click the **Open in Colab** badge, then **Runtime → Change runtime type → T4 GPU**.
2. Accept the competition rules on Kaggle (step 2 above) and create a Kaggle API token at
   [kaggle.com/settings](https://www.kaggle.com/settings) → **Create New Token**. The notebook will ask for it.

**How to run a cell:** click inside it and press **Shift+Enter**. A cell is finished when the `[ ]` on its left shows a number.
Run the cells from top to bottom. If something goes wrong, use **Run → Restart & clear outputs** (Kaggle) or
**Runtime → Restart session** (Colab) and start again from the first cell.

## Watch first

Two videos before you start. They are optional, but they make the rest of the lesson easier to follow.
You do not need to understand everything in them.

**Practical Deep Learning for Coders, lesson 1** (fast.ai, Jeremy Howard). Run the cell below to play it here, or [open it on YouTube](https://www.youtube.com/watch?v=8SF_h3xF3cE).
The video is long; the first 30 minutes are enough.

In [ ]:
# Run this cell to play the video here.
from IPython.display import IFrame
IFrame("https://www.youtube-nocookie.com/embed/8SF_h3xF3cE", width=720, height=405)

*Why watch it:* this is the course that inspired ours. Jeremy builds a model that tells photos of birds from photos of forests,
following the same steps you will follow today with lymph node patches.

*What to focus on:*

- How few steps it takes: collect pictures, label them, train a model, check the result.
- His point that you do not need advanced maths, huge amounts of data or expensive computers to start.
- The pictures of what a network "sees" at each layer: simple edges and colours first, then textures, then whole objects.
  Think of it as going from staining pattern, to cells, to tissue architecture.

*After watching you should be able to say,* in one sentence, what "training a model" means.

**But what is a neural network?** (3Blue1Brown, about 20 minutes). Run the cell below to play it here, or [open it on YouTube](https://www.youtube.com/watch?v=aircAruvnKk).

In [ ]:
# Run this cell to play the video here.
from IPython.display import IFrame
IFrame("https://www.youtube-nocookie.com/embed/aircAruvnKk", width=720, height=405)

*Why watch it:* the first video shows what a model can do. This one shows what is inside it, with clear animations and no coding.

*What to focus on:*

- A neural network is a stack of **layers**. Each layer is just a set of numbers.
- Each layer builds on the one before: pixels, then edges, then shapes, then the answer (here, a handwritten digit).
- **Training** means slowly adjusting millions of small numbers, called **weights**, until the answers come out right.
- When formulas appear, let them pass. The pictures carry the idea.

*After watching you should be able to explain* what a "layer" and a "weight" are in your own words,
and why a network needs many labelled examples to learn.

## Step 0: Set up

The first cell installs the tools and checks that you have a GPU. Run it now.

In [ ]:
# COMPPATH-SETUP
# Run this cell first: click inside it, then press Shift+Enter.
# It installs the tools this lesson needs and works out where it is running.
# You do not need to read or change anything here.
import os
import subprocess
import sys
from pathlib import Path

# Exact versions, so this notebook keeps working for years (see environment.txt).
PACKAGES = [
    "fastai==2.8.12",
    "fastcore==2.2.33",
    "fasttransform==0.0.5",
    "kagglehub==1.0.3",
]
if PACKAGES and os.environ.get("COMPPATH_SKIP_INSTALL", "0") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES], check=True)

def detect_environment():
    """Return "kaggle", "colab" or "local". Mirrors utils/env.py in the course repository."""
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").exists():
        return "kaggle"
    if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
        return "colab"
    return "local"


ENV = detect_environment()
SMOKE_TEST = os.environ.get("COMPPATH_SMOKE_TEST", "0") == "1"  # used by automated tests only
if ENV == "kaggle":
    WORK_DIR = Path("/kaggle/working")
elif ENV == "colab":
    WORK_DIR = Path("/content")
else:
    WORK_DIR = Path(os.environ.get("COMPPATH_WORK_DIR", Path.cwd() / "work"))
WORK_DIR.mkdir(parents=True, exist_ok=True)

import torch  # PyTorch is already installed on Kaggle and Colab

HAS_GPU = torch.cuda.is_available()
if not HAS_GPU and not SMOKE_TEST:
    print("⚠️  No GPU found. Training will be very slow.")
    print("   Kaggle: Settings → Accelerator → GPU T4 x2.  Colab: Runtime → Change runtime type → T4 GPU.")
    print("   Then restart and run this cell again.")

print(f"Running on: {ENV}. Files you create will be saved in: {WORK_DIR}")

## Step 1: Meet the data

We use **PatchCamelyon** (PCam): small 96 × 96 pixel patches cut from H&E-stained sections of sentinel lymph nodes
from breast cancer patients. The slides come from the CAMELYON16 challenge, scanned at two Dutch hospitals (Radboud UMC and UMC Utrecht). The patches were cut at 10× magnification.

Each patch has a label:

- **tumor** (`1`): the central 32 × 32 pixel square contains at least one pixel of metastatic tumor, as outlined by pathologists
- **normal** (`0`): no tumor in the central square

So "normal" here does not mean "normal lymph node". It means "no tumor in the middle of this little square".
Labels always carry assumptions like this, and asking what exactly a label means is one of the most useful habits in this field.

The next cell finds the images. On Kaggle it should take a second; on Colab it downloads about 7 GB first.

In [ ]:
COMPETITION = "histopathologic-cancer-detection"

HELP = f"""
Could not find the PatchCamelyon images. Please check:
  1. You accepted the rules on https://www.kaggle.com/competitions/{COMPETITION}/rules
     (click "Late Submission" or "Join Competition" while logged in).
  2. On Kaggle: right-hand panel → "Add Input" → Competitions → search "{COMPETITION}" → Add.
     Internet must be ON (Settings → Internet), which needs a verified phone number.
  3. On Colab or your own computer: the cell asks you to log in to Kaggle. Use an API token
     from https://www.kaggle.com/settings ("Create New Token").
"""


def make_synthetic_patches(folder, n=64, seed=0):
    """For automated tests only: makes fake pink and purple squares instead of real patches."""
    import numpy as np
    import pandas as pd
    from PIL import Image

    rng = np.random.default_rng(seed)
    (folder / "train").mkdir(parents=True, exist_ok=True)
    rows = []
    for i in range(n):
        label = i % 2
        colour = np.array([110, 50, 140]) if label else np.array([235, 175, 210])
        pixels = np.clip(colour + rng.normal(0, 30, (96, 96, 3)), 0, 255).astype("uint8")
        Image.fromarray(pixels).save(folder / "train" / f"synthetic{i:04d}.tif")
        rows.append({"id": f"synthetic{i:04d}", "label": label})
    pd.DataFrame(rows).to_csv(folder / "train_labels.csv", index=False)
    return folder


def find_patchcamelyon():
    """Return the folder that holds train_labels.csv and the train/ images."""
    if SMOKE_TEST:
        return make_synthetic_patches(WORK_DIR / "synthetic-patches")

    # 1. Is the data already attached to this notebook? (Kaggle: "Add Input")
    inputs = Path("/kaggle/input")
    if inputs.exists():
        found = list(inputs.glob("*/train_labels.csv")) + list(inputs.glob("*/*/train_labels.csv"))
        if found:
            return found[0].parent

    # 2. Otherwise download it from Kaggle (about 7 GB, takes a few minutes).
    import kagglehub

    if ENV != "kaggle":
        try:
            kagglehub.whoami()
        except Exception:
            kagglehub.login()  # shows a box where you paste your Kaggle API token
    try:
        return Path(kagglehub.competition_download(COMPETITION))
    except Exception as error:
        raise RuntimeError(HELP) from error


DATA_DIR = find_patchcamelyon()
print("Found the data in:", DATA_DIR)

In [ ]:
import pandas as pd

labels = pd.read_csv(DATA_DIR / "train_labels.csv")
labels["diagnosis"] = labels["label"].map({0: "normal", 1: "tumor"})
print(f"{len(labels):,} labelled patches in total")
labels["diagnosis"].value_counts()

About 220,000 patches, with more normal than tumor. Training on all of them would take an hour or more,
so we pick a smaller, balanced subset. The next cell holds every setting for this lesson.
Leave it as it is for your first run; the homework asks you to experiment with it.

In [ ]:
# ⚙️  The settings for this lesson. This is the only cell you are invited to change.
N_PATCHES = 20_000      # how many patches to use (half tumor, half normal). The full set has about 220,000.
VALID_FRACTION = 0.2    # share of patches held back to test the model (it never trains on these)
ARCHITECTURE = "resnet34"  # try "resnet18" (smaller, faster) or "resnet50" (bigger, slower)
EPOCHS = 3              # how many times the model sees every training patch
BATCH_SIZE = 128        # how many patches the GPU looks at in one go
SEED = 42               # makes the random choices repeatable

if SMOKE_TEST:  # tiny settings for automated tests on a CPU
    N_PATCHES, ARCHITECTURE, EPOCHS, BATCH_SIZE = 64, "resnet18", 1, 16

Now we split our patches into two groups:

- the **training set**, which the model learns from
- the **validation set**, which the model never sees during training. We use it to check how well it does,
  like an exam with questions the student has not seen before.

In [ ]:
# Pick N_PATCHES patches, half from each class, then hold back VALID_FRACTION of each class for testing.
per_class = min(N_PATCHES // 2, labels["diagnosis"].value_counts().min())
patches = labels.groupby("diagnosis", group_keys=False).sample(n=per_class, random_state=SEED)
patches["fname"] = "train/" + patches["id"] + ".tif"
validation = patches.groupby("diagnosis", group_keys=False).sample(frac=VALID_FRACTION, random_state=SEED)
patches["is_valid"] = patches.index.isin(validation.index)

pd.crosstab(patches["diagnosis"], patches["is_valid"].map({False: "training", True: "validation"}))

## Step 2: Look at the data like a pathologist

The next cell prepares the patches for the model and shows you a random handful with their labels.
Before you go on, look at them closely. Can you see why a patch is called tumor? Would you agree with every label?

The images are randomly flipped and rotated each time the model sees them (*data augmentation*).
A metastasis is still a metastasis upside down, and showing the model rotated copies teaches it that orientation does not matter.
Lesson 6 covers augmentation properly.

In [ ]:
from fastai.vision.all import *  # loads the fastai toolbox for images

set_seed(SEED, reproducible=True)
dls = ImageDataLoaders.from_df(
    patches,
    path=DATA_DIR,
    fn_col="fname",
    label_col="diagnosis",
    valid_col="is_valid",
    batch_tfms=aug_transforms(flip_vert=True, max_rotate=90, max_zoom=1.0, max_warp=0, max_lighting=0.1),
    bs=BATCH_SIZE,
    num_workers=0 if SMOKE_TEST else None,
)
dls.show_batch(max_n=16, nrows=4, figsize=(8, 8))

## Step 3: Choose a model

We do not start from scratch. We take a **ResNet**, a neural network that has already been trained on
over a million everyday photos (the ImageNet dataset), and adapt it to histology. This is called **transfer learning**.

Think of a resident who arrives with good eyes for shape, colour and texture from years of looking at the world,
but has never seen a lymph node. They do not need to learn to see from scratch, only what matters in this tissue.

In [ ]:
ARCHITECTURES = {"resnet18": resnet18, "resnet34": resnet34, "resnet50": resnet50}

learn = vision_learner(dls, ARCHITECTURES[ARCHITECTURE], metrics=[accuracy, RocAucBinary()])
print(f"Loaded a pretrained {ARCHITECTURE}. The classes it will learn are: {list(dls.vocab)}")

## Step 4: Train

Now the model learns. `fine_tune` first trains only the new last layer for one round, then gently adjusts the whole network for `EPOCHS` rounds.
An **epoch** is one pass through every training patch.

After each epoch you will see a row in a table:

| Column | What it means |
|---|---|
| `train_loss` | how wrong the model is on the training patches (lower is better) |
| `valid_loss` | how wrong it is on the validation patches it has never trained on |
| `accuracy` | share of validation patches it labels correctly |
| `roc_auc_score` | how well it ranks tumor patches above normal ones, from 0.5 (coin toss) to 1.0 (perfect) |
| `time` | how long the epoch took |

This takes about 5 to 10 minutes on a Kaggle GPU. Good time for a coffee.

In [ ]:
import time

start = time.perf_counter()
learn.fine_tune(EPOCHS)
minutes = (time.perf_counter() - start) / 60
print(f"Training took {minutes:.1f} minutes.")

If everything went well, accuracy should be somewhere around 90% and AUC above 0.95.
**You have just trained a deep learning model on histopathology.** Congratulations!

## Step 5: How good is it, really?

Accuracy is one number. As pathologists we want to know *which* mistakes the model makes.
A **confusion matrix** counts them: rows are the true diagnosis, columns are the model's answer.

- **Sensitivity**: of all tumor patches, how many did the model catch? A missed metastasis is a false negative.
- **Specificity**: of all normal patches, how many did it correctly call normal? A false alarm is a false positive.

Which of the two errors would worry you more if this model screened sentinel nodes?

> **Explained: confusion matrix**
>
> You already know this table. It is the **2 × 2 table** from diagnostic test studies, the one you use to work out sensitivity and specificity.
>
> The model has looked at every validation patch. For each patch we compare its answer with the true label,
> and put the patch into one of four boxes:
>
> | | Model says **normal** | Model says **tumor** |
> |---|---|---|
> | **Truly normal** | Correct (true negative) | False alarm (false positive) |
> | **Truly tumor** | Missed tumor (false negative) | Correct (true positive) |
>
> Then we count how many patches landed in each box. That is all a confusion matrix is.
>
> - The two **correct** boxes run from top left to bottom right. You want big numbers there.
> - The other two boxes are the **mistakes**, split by type: false alarms and missed tumors.
> - The name simply means: it shows where the model gets *confused*.

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(4, 4))

# Rows are the true diagnosis, columns are the model's answer, in the order of dls.vocab: normal, tumor.
(tn, fp), (fn, tp) = interp.confusion_matrix()
print(f"Sensitivity (tumor patches the model caught):      {tp / (tp + fn):.1%}")
print(f"Specificity (normal patches correctly called normal): {tn / (tn + fp):.1%}")

## Step 6: Look at the mistakes

This is where your training as a pathologist matters most. The cell below shows the nine patches the model got
*most* wrong, with four values over each: the model's answer, the true label, the loss (how wrong it was)
and the probability the model gave to the true class.

For each one, ask yourself:

- Is the **label** wrong? Annotations are drawn by hand, often at low magnification.
- Is the tumor only at the **edge** of the patch, outside the central square?
- Is there an **artefact**: blur, folds, ink, crush, out-of-focus areas?
- Would **you** be able to decide from this tiny square alone?

In [ ]:
interp.plot_top_losses(9, nrows=3, figsize=(10, 10))

## Step 7: Ask the model about one patch

Here the model looks at a single validation patch and gives its answer with a probability.
Run the cell several times after changing `random_state=SEED` to another number to see different patches.

In [ ]:
example = patches[patches["is_valid"]].sample(1, random_state=SEED).iloc[0]
image = PILImage.create(DATA_DIR / example["fname"])
answer, _, probabilities = learn.predict(image)

show_image(image, figsize=(3, 3), title=f"True: {example['diagnosis']}")
print(f"The model says: {answer} (probability of tumor: {probabilities[1]:.1%})")

Finally, save the trained model. On Kaggle it appears under **Output** in the right-hand panel, and you can download it.

In [ ]:
model_file = WORK_DIR / "lesson01-patchcamelyon.pkl"
learn.export(model_file)
print("Saved your model to:", model_file)

## A word of caution

The numbers above are **too optimistic**, and it is important to know why.

Many patches come from the same slide, and therefore the same patient. Our split put patches at random into training and validation,
so the model was probably tested on neighbours of patches it trained on. That is like testing a student on cases from the
same patient they studied. The Kaggle version of PCam does not tell us which patient each patch came from, so we cannot fix it here.
**Lesson 5** is all about this problem, called *data leakage*, and about splitting by patient.

This model is a teaching exercise, not a diagnostic tool. Never use it on real patients.

## Recap

In about 15 minutes, without writing code, you:

1. loaded about 20,000 labelled H&E patches,
2. adapted a network pretrained on everyday photos to lymph node histology,
3. measured its accuracy, AUC, sensitivity and specificity,
4. reviewed its worst mistakes like a pathologist.

Every box we opened today without explanation, such as the data, the split, the model, the training and the metrics,
gets its own lesson later. In **lesson 2** we step back to the glass slide and the scanner, and in **lesson 3** we read this
notebook line by line.

## Now try it yourself

**Homework: experiment, observe and share.**

1. **Change one setting at a time** in the ⚙️ settings cell, then use **Run All** and record the results in a table like this one:

   | Run | N_PATCHES | ARCHITECTURE | EPOCHS | Accuracy | AUC | Sensitivity | Specificity | Minutes |
   |---|---|---|---|---|---|---|---|---|
   | 1 (default) | 20,000 | resnet34 | 3 | | | | | |
   | 2 | 5,000 | resnet34 | 3 | | | | | |
   | 3 | 20,000 | resnet18 | 3 | | | | | |
   | 4 | 20,000 | resnet34 | 1 | | | | | |

   Try a bigger run of your choice too (for example `N_PATCHES = 60_000`). Watch your GPU quota on Kaggle: the counter is at the top of the right-hand panel.

2. **Be the pathologist.** Pick three of the top-loss patches and write two or three sentences on each: do you agree with the label or with the model, and why?

3. **Think about your own practice.** Write a short paragraph: which task in your department could a patch classifier like this help with? What would the "label" be, and who would draw it?

4. **Share it.** Put your table and notes in a markdown cell at the end of this notebook (double-click this cell to see how markdown looks).
   On Kaggle, click **Save Version**, then **Share → Public**, and post the link to your cohort.
   This is the first piece of your portfolio.

When you are done, check your work against `checkpoint.md` in this lesson's folder.

## Further reading and credits

- **Data:** PatchCamelyon by Bas Veeling and colleagues, released under CC0.
  B. S. Veeling, J. Linmans, J. Winkens, T. Cohen, M. Welling. *Rotation Equivariant CNNs for Digital Pathology.* MICCAI 2018. [arXiv:1806.03962](https://arxiv.org/abs/1806.03962).
  The Kaggle version used here ([Histopathologic Cancer Detection](https://www.kaggle.com/competitions/histopathologic-cancer-detection)) removes duplicate patches.
- **Original slides:** B. Ehteshami Bejnordi et al. *Diagnostic Assessment of Deep Learning Algorithms for Detection of Lymph Node Metastases in Women With Breast Cancer.* JAMA 2017;318(22):2199–2210.
- **Software:** [fastai](https://docs.fast.ai) by Jeremy Howard, Sylvain Gugger and contributors; [PyTorch](https://pytorch.org).
- **Further watching:** the full [Practical Deep Learning for Coders](https://course.fast.ai) course.